# JR_train_snow 誤差要因分析

現行パイプラインで検証予測を作り、**WMAEへの寄与が大きい誤差が、どの条件で発生しているか**を調べるNotebookです。

## 方針
1. 現行の特徴量生成・時系列split・LightGBMを再現
2. 行ごとのWMAE寄与を計算
3. 誤差上位ケースを確認
4. 実測値の大きさ別に誤差を確認
5. 月・曜日・列車番号などで誤差を比較
6. 既存の気象特徴量と誤差の関係を見る

**このNotebookでは新しい特徴量を追加しません。**
まず「何を外しているのか」を特定し、その結果を次の特徴量設計につなげます。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

PROJECT_ROOT = Path("/home/keiseki/JR_train_snow")
SRC_ROOT = PROJECT_ROOT / "30.src"
CONFIG_DIR = PROJECT_ROOT / "00.config"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from jr_snow.data import load_train_test_data
from jr_snow.features import prepare_model_inputs
from jr_snow.modeling import train_lightgbm_model

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


## 1. 設定・データ読み込み

In [ ]:
with open(CONFIG_DIR / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

with open(CONFIG_DIR / "path.yaml", encoding="utf-8") as f:
    path_config = yaml.safe_load(f)

train_path = Path(path_config["INTERIUM_PATH"]["train_data"])
test_path = Path(path_config["INTERIUM_PATH"]["test_data"])

print("train:", train_path)
print("test :", test_path)
print("split_date:", config["split_date"])

df_train, df_test = load_train_test_data(
    train_data_path=train_path,
    test_data_path=test_path,
)

print("train shape:", df_train.shape)
print("test shape :", df_test.shape)


In [ ]:
feature_columns = {
    "feature_list": config["FEATURE"]["FEATURE_LIST"],
    "categorical_cols": config["FEATURE"]["CATEGORICAL_COLS"],
}

prepared = prepare_model_inputs(
    df_train=df_train.copy(),
    df_test=df_test.copy(),
    feature_columns=feature_columns,
    split_date=config["split_date"],
    target_col="合計",
)

X_train = prepared["X_train"]
X_valid = prepared["X_valid"]
y_train = prepared["y_train"]
y_valid = prepared["y_valid"]
feature_list = prepared["feature_list"]
categorical_cols = prepared["categorical_cols"]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("features:", len(feature_list))
print("categorical:", categorical_cols)


## 2. 現行モデルで検証予測

`config.yaml` の現在の `MODEL_PARAMS` を使用します。


In [ ]:
model = train_lightgbm_model(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    categorical_cols=categorical_cols,
    model_params=config["MODEL_PARAMS"].copy(),
    num_boost_round=1000,
    early_stopping_rounds=100,
)

y_pred = np.asarray(model.predict(X_valid), dtype=float)

print("prediction min :", y_pred.min())
print("prediction max :", y_pred.max())
print("prediction mean:", y_pred.mean())


## 3. WMAEと行ごとの誤差寄与

現在の評価式

`weight = 1 + 10000 * abs(y_true)`

に合わせて、各行の `weight × absolute error` を計算します。

**今回の分析では、この「WMAE寄与」が最重要です。**


In [ ]:
error_df = X_valid.copy().reset_index(drop=True)
y_true_arr = np.asarray(y_valid.reset_index(drop=True), dtype=float)

error_df["実測"] = y_true_arr
error_df["予測"] = y_pred
error_df["絶対誤差"] = np.abs(error_df["実測"] - error_df["予測"])
error_df["符号付き誤差"] = error_df["予測"] - error_df["実測"]
error_df["WMAE重み"] = 1 + 10000 * np.abs(error_df["実測"])
error_df["WMAE寄与"] = error_df["絶対誤差"] * error_df["WMAE重み"]

wmae = error_df["WMAE寄与"].sum() / error_df["WMAE重み"].sum()
print(f"Validation WMAE = {wmae:.8f}")

error_df.sort_values("WMAE寄与", ascending=False).head(30)


## 4. 誤差上位ケース

In [ ]:
top_errors = error_df.sort_values("WMAE寄与", ascending=False).head(50).copy()

show_cols = [
    c for c in [
        "年月日", "年", "月", "日", "曜日",
        "列車番号", "金沢着雪ゼロ列車フラグ",
        "実測", "予測", "絶対誤差", "符号付き誤差",
        "WMAE重み", "WMAE寄与",
    ]
    if c in top_errors.columns
]

top_errors[show_cols]


## 5. 実測値の大きさ別に誤差を見る

In [ ]:
bins = [-np.inf, 0, 0.01, 0.1, 0.5, 1, 2, 5, 10, 20, 50, np.inf]
labels = ["0", "0-0.01", "0.01-0.1", "0.1-0.5", "0.5-1", "1-2",
          "2-5", "5-10", "10-20", "20-50", "50+"]

error_df["実測区分"] = pd.cut(
    error_df["実測"], bins=bins, labels=labels, right=True
)

target_summary = error_df.groupby("実測区分", observed=False).agg(
    件数=("実測", "size"),
    実測平均=("実測", "mean"),
    予測平均=("予測", "mean"),
    絶対誤差平均=("絶対誤差", "mean"),
    WMAE寄与合計=("WMAE寄与", "sum"),
).reset_index()

target_summary["WMAE寄与割合"] = (
    target_summary["WMAE寄与合計"] / error_df["WMAE寄与"].sum()
)

target_summary


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(target_summary["実測区分"].astype(str), target_summary["WMAE寄与割合"])
plt.xticks(rotation=45)
plt.ylabel("WMAE contribution ratio")
plt.xlabel("Actual snow amount")
plt.title("WMAE contribution by actual snow amount")
plt.tight_layout()
plt.show()


## 6. 過小予測・過大予測

In [ ]:
error_df["予測方向"] = np.where(
    error_df["予測"] >= error_df["実測"],
    "過大予測",
    "過小予測",
)

direction_summary = error_df.groupby("予測方向").agg(
    件数=("実測", "size"),
    実測平均=("実測", "mean"),
    絶対誤差平均=("絶対誤差", "mean"),
    WMAE寄与合計=("WMAE寄与", "sum"),
).reset_index()

direction_summary["WMAE寄与割合"] = (
    direction_summary["WMAE寄与合計"] / error_df["WMAE寄与"].sum()
)

direction_summary


## 7. 月・曜日による誤差の偏り

In [ ]:
def group_error_summary(df, group_col):
    out = df.groupby(group_col, dropna=False).agg(
        件数=("実測", "size"),
        実測平均=("実測", "mean"),
        予測平均=("予測", "mean"),
        絶対誤差平均=("絶対誤差", "mean"),
        WMAE寄与合計=("WMAE寄与", "sum"),
    ).reset_index()
    out["WMAE寄与割合"] = (
        out["WMAE寄与合計"] / df["WMAE寄与"].sum()
    )
    return out.sort_values("WMAE寄与合計", ascending=False)

if "月" in error_df.columns:
    display(group_error_summary(error_df, "月"))

if "曜日" in error_df.columns:
    display(group_error_summary(error_df, "曜日"))


## 8. 列車番号による誤差の偏り

列車番号由来特徴量を追加しても全体スコアがほぼ動かなかったため、
**特定列車だけ極端に外しているのか**を確認します。


In [ ]:
if "列車番号" in error_df.columns:
    train_summary = error_df.groupby("列車番号", dropna=False).agg(
        件数=("実測", "size"),
        実測平均=("実測", "mean"),
        予測平均=("予測", "mean"),
        絶対誤差平均=("絶対誤差", "mean"),
        WMAE寄与合計=("WMAE寄与", "sum"),
    ).reset_index()

    train_summary["WMAE寄与割合"] = (
        train_summary["WMAE寄与合計"] / error_df["WMAE寄与"].sum()
    )

    display(
        train_summary
        .sort_values("WMAE寄与合計", ascending=False)
        .head(30)
    )


## 9. 実測値 vs 予測値

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(error_df["実測"], error_df["予測"], alpha=0.35, s=15)

max_value = max(error_df["実測"].max(), error_df["予測"].max())
plt.plot([0, max_value], [0, max_value], linestyle="--")

plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs predicted")
plt.tight_layout()
plt.show()


## 10. 大きな誤差ケースの気象特徴量

新しい特徴量は作らず、**現在モデルへ投入している気象特徴量**について、
誤差上位5%と検証データ全体の平均を比較します。

差が大きい項目があれば、次の特徴量設計候補です。


In [ ]:
weather_cols = [
    c for c in error_df.columns
    if any(key in str(c) for key in [
        "気温", "降水量", "降雪", "積雪", "日照", "日射",
        "風速", "風向", "湿度", "露点", "天気", "気圧", "雲量"
    ])
]

numeric_weather_cols = [
    c for c in weather_cols
    if pd.api.types.is_numeric_dtype(error_df[c])
]

top_n = max(10, int(len(error_df) * 0.05))
top_error_df = error_df.nlargest(top_n, "WMAE寄与")

weather_compare = pd.DataFrame({
    "全検証": error_df[numeric_weather_cols].mean(numeric_only=True),
    "誤差上位5%": top_error_df[numeric_weather_cols].mean(numeric_only=True),
})

weather_compare["差"] = (
    weather_compare["誤差上位5%"] - weather_compare["全検証"]
)
weather_compare["差の絶対値"] = weather_compare["差"].abs()

weather_compare.sort_values("差の絶対値", ascending=False).head(30)


## 11. 大きな着雪を外したケース

In [ ]:
large_snow_misses = error_df[
    (error_df["実測"] > error_df["実測"].quantile(0.95))
    & (error_df["予測"] < error_df["実測"])
].sort_values("WMAE寄与", ascending=False)

extra_cols = [c for c in numeric_weather_cols if c not in show_cols]

large_snow_misses[show_cols + extra_cols].head(30)


## 12. 次の分析へ

このNotebookで特に確認するポイント:

- WMAE寄与上位ケースに共通する条件
- 大きな着雪を過小予測しているか
- 特定の月・時刻・列車番号に誤差が集中しているか
- 特定地域の気象条件で誤差が増えているか
- 現在の気象特徴量では説明できないケースがあるか

ここで**具体的な誤差パターンを1つ見つけてから**、そのパターンを特徴量化します。
